# 📧 Spam Email Detection Using TensorFlow & LSTM

A complete machine-learning project for classifying emails as **Spam** or **Ham (Not Spam)** using Natural Language Processing (NLP) and a TensorFlow/Keras LSTM neural network.

### Dataset
The uploaded dataset is `spam_ham_dataset.csv`.

**Dataset size:** 5,171 rows × 4 columns  
**Main text column:** `text`  
**Target column:** `label` (`spam` / `ham`)

### Workflow
1. Import libraries
2. Load and inspect the dataset
3. Explore class distribution
4. Balance the classes
5. Clean email text
6. Visualize common words
7. Tokenize and pad sequences
8. Build an LSTM model
9. Train with callbacks
10. Evaluate the model
11. Plot training performance
12. Test the model on new emails
13. Save the trained model

In [ ]:
# Optional: install dependencies if they are missing
# Uncomment and run if required.

# %pip install pandas numpy matplotlib seaborn nltk wordcloud tensorflow scikit-learn

In [ ]:
import re
import string
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import nltk
from nltk.corpus import stopwords
from wordcloud import WordCloud

import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Embedding, LSTM, Dense
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

nltk.download("stopwords")

print("TensorFlow version:", tf.__version__)

In [ ]:
# Load the uploaded dataset
file_path = "spam_ham_dataset.csv"
data = pd.read_csv(file_path)

print("Dataset loaded successfully!")
print("Shape:", data.shape)
display(data.head())

In [ ]:
# Dataset information
print("Columns:")
print(data.columns.tolist())

print("\nMissing values:")
display(data.isnull().sum())

print("\nData types:")
display(data.dtypes)

In [ ]:
# Select the expected columns
text_col = "text"
label_col = "label"

if text_col not in data.columns or label_col not in data.columns:
    raise ValueError(
        f"Expected columns '{text_col}' and '{label_col}'. "
        f"Available columns: {data.columns.tolist()}"
    )

data = data[[text_col, label_col]].copy()
data[text_col] = data[text_col].fillna("").astype(str)
data[label_col] = data[label_col].astype(str).str.lower().str.strip()

print(data[label_col].value_counts())

In [ ]:
# Visualize the class distribution
plt.figure(figsize=(7, 5))
sns.countplot(data=data, x=label_col)
plt.title("Spam vs Ham Email Distribution")
plt.xlabel("Email Label")
plt.ylabel("Count")
plt.show()

## 1. Balance the Dataset

The source workflow balances the classes by downsampling the majority class so that spam and ham have the same number of examples.

In [ ]:
# Balance the classes
spam_data = data[data[label_col] == "spam"]
ham_data = data[data[label_col] == "ham"]

min_count = min(len(spam_data), len(ham_data))

spam_data = spam_data.sample(n=min_count, random_state=42)
ham_data = ham_data.sample(n=min_count, random_state=42)

balanced_data = pd.concat([spam_data, ham_data])
balanced_data = balanced_data.sample(frac=1, random_state=42).reset_index(drop=True)

print("Balanced dataset shape:", balanced_data.shape)
print(balanced_data[label_col].value_counts())

In [ ]:
# Visualize balanced classes
plt.figure(figsize=(7, 5))
sns.countplot(data=balanced_data, x=label_col)
plt.title("Balanced Spam vs Ham Distribution")
plt.xlabel("Email Label")
plt.ylabel("Count")
plt.show()

## 2. Text Cleaning

Email text is normalized by:
- converting to lowercase
- removing punctuation
- removing English stopwords
- removing extra whitespace

In [ ]:
stop_words = set(stopwords.words("english"))

def clean_text(text):
    text = text.lower()
    text = text.translate(str.maketrans("", "", string.punctuation))
    words = text.split()
    words = [word for word in words if word not in stop_words]
    return " ".join(words)

balanced_data["clean_text"] = balanced_data[text_col].apply(clean_text)

display(balanced_data[[text_col, "clean_text", label_col]].head())

In [ ]:
# Word cloud for spam emails
spam_text = " ".join(
    balanced_data.loc[balanced_data[label_col] == "spam", "clean_text"]
)

spam_wordcloud = WordCloud(
    width=1000,
    height=500,
    background_color="white"
).generate(spam_text)

plt.figure(figsize=(12, 6))
plt.imshow(spam_wordcloud, interpolation="bilinear")
plt.axis("off")
plt.title("Common Words in Spam Emails")
plt.show()

In [ ]:
# Word cloud for ham emails
ham_text = " ".join(
    balanced_data.loc[balanced_data[label_col] == "ham", "clean_text"]
)

ham_wordcloud = WordCloud(
    width=1000,
    height=500,
    background_color="white"
).generate(ham_text)

plt.figure(figsize=(12, 6))
plt.imshow(ham_wordcloud, interpolation="bilinear")
plt.axis("off")
plt.title("Common Words in Ham Emails")
plt.show()

## 3. Train/Test Split and Tokenization

The cleaned emails are converted into integer sequences using a Keras `Tokenizer`. Sequences are padded to a maximum length of **100** tokens.

In [ ]:
# Encode labels
balanced_data["label_num"] = balanced_data[label_col].map({"ham": 0, "spam": 1})

X = balanced_data["clean_text"].values
y = balanced_data["label_num"].values

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

In [ ]:
# Tokenization
max_words = 10000
max_len = 100

tokenizer = Tokenizer(num_words=max_words, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

X_train_pad = pad_sequences(
    X_train_seq,
    maxlen=max_len,
    padding="post",
    truncating="post"
)

X_test_pad = pad_sequences(
    X_test_seq,
    maxlen=max_len,
    padding="post",
    truncating="post"
)

vocab_size = min(max_words, len(tokenizer.word_index) + 1)

print("Vocabulary size:", vocab_size)
print("Training tensor shape:", X_train_pad.shape)
print("Testing tensor shape:", X_test_pad.shape)

## 4. Build the LSTM Model

Architecture:
- Embedding layer
- LSTM layer with 16 units
- Dense layer with 32 ReLU units
- Output Dense layer with sigmoid activation

The sigmoid output represents the probability of the email being spam.

In [ ]:
# Build the TensorFlow/Keras model
model = Sequential([
    Embedding(input_dim=vocab_size, output_dim=64, input_length=max_len),
    LSTM(16),
    Dense(32, activation="relu"),
    Dense(1, activation="sigmoid")
])

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model.summary()

## 5. Train the Model

`EarlyStopping` helps prevent unnecessary training when validation loss stops improving, while `ReduceLROnPlateau` lowers the learning rate when progress slows.

In [ ]:
early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True
)

reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=2,
    min_lr=1e-6
)

history = model.fit(
    X_train_pad,
    y_train,
    validation_split=0.2,
    epochs=20,
    batch_size=32,
    callbacks=[early_stopping, reduce_lr],
    verbose=1
)

## 6. Evaluate the Model

In [ ]:
# Evaluate on the held-out test set
test_loss, test_accuracy = model.evaluate(X_test_pad, y_test, verbose=0)

print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")

In [ ]:
# Classification report
y_prob = model.predict(X_test_pad, verbose=0).ravel()
y_pred = (y_prob >= 0.5).astype(int)

print(classification_report(
    y_test,
    y_pred,
    target_names=["Ham", "Spam"]
))

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    xticklabels=["Ham", "Spam"],
    yticklabels=["Ham", "Spam"]
)
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

In [ ]:
# Training and validation accuracy
plt.figure(figsize=(10, 5))
plt.plot(history.history["accuracy"], label="Training Accuracy")
plt.plot(history.history["val_accuracy"], label="Validation Accuracy")
plt.title("Training vs Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.show()

In [ ]:
# Training and validation loss
plt.figure(figsize=(10, 5))
plt.plot(history.history["loss"], label="Training Loss")
plt.plot(history.history["val_loss"], label="Validation Loss")
plt.title("Training vs Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.show()

## 7. Test the Model With a New Email

Enter any email message below. The trained model will classify it as **Spam** or **Ham** and display the predicted spam probability.

In [ ]:
def predict_email(email_text):
    cleaned = clean_text(email_text)
    sequence = tokenizer.texts_to_sequences([cleaned])
    padded = pad_sequences(
        sequence,
        maxlen=max_len,
        padding="post",
        truncating="post"
    )

    probability = float(model.predict(padded, verbose=0)[0][0])
    prediction = "Spam" if probability >= 0.5 else "Ham"

    return prediction, probability

# Example
sample_email = "Congratulations! You have won a free cash prize. Click the link now!"
prediction, probability = predict_email(sample_email)

print("Prediction:", prediction)
print(f"Spam Probability: {probability:.2%}")

In [ ]:
# Try your own email
my_email = input("Enter an email message: ")

prediction, probability = predict_email(my_email)

print("\nPrediction:", prediction)
print(f"Spam Probability: {probability:.2%}")
print(f"Ham Probability: {(1 - probability):.2%}")

## 8. Save the Model

The trained model is saved in Keras format so it can be loaded later without retraining.

In [ ]:
model.save("spam_email_lstm.keras")

print("Model saved as: spam_email_lstm.keras")

## Project Summary

This project demonstrates an end-to-end NLP classification pipeline:

**Email Dataset → Text Cleaning → Tokenization → Padding → LSTM → Classification → Evaluation**

The model can classify incoming email text into:
- **Ham:** legitimate/non-spam email
- **Spam:** unwanted/spam email

> **Note:** The final accuracy depends on the uploaded dataset, random split, TensorFlow version, preprocessing, and training environment. The notebook reports the accuracy obtained when you run it rather than assuming a fixed result.